# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imhs14/flyrank-ml-internship-starter-main/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I've choose the Clustering Lane, because it is new to me and there are no right answers in this, to compare. comparing to others this one is unsupervised learning no labels, no right answers to check against, new algorithms K-Means, new prep step (feature scaling), new visualization method (PCA)

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, pandas as pd, numpy as np

while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

reading = pd.read_csv("data/raw/content_refresh_anonymized.csv")
size_of_data = np.shape(reading)
print(size_of_data)
b = reading.select_dtypes(include="number").columns
print(b)


(30000, 44)
Index(['search_volume', 'competition', 'cpc', 'word_count', 'char_count',
       'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
       'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d',
       'scroll_events_90d', 'days_with_impressions', 'days_with_sessions',
       'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
       'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
       'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr',
       'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
       'trend_pct'],
      dtype='str')


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

##This work will help decide which of ~30,000 pages the content team should review first. A content strategist would use the cluster labels to prioritize their weekly review queue instead of checking pages one by one. If a cluster is mislabeled — e.g., a declining group gets treated as healthy — the cost is wasted editor time on the wrong pages, and real declining pages get missed and keep losing traffic.

Since clustering has no ground truth to check against, mistakes are sneaky they don't throw an error, they just quietly produce a plausible-looking but wrong story. concretely.

Naming clusters before inspecting them. If you decide "Cluster 1 = champions" just because it sounds like the biggest group, without actually checking its real numbers, you might mislabel a genuinely struggling group as healthy — and the team protects pages that actually needed help.

Not scaling features. If you skip this, your clusters might just silently reflect "high word count vs. low word count" and nothing else — you'd think you found 4 meaningful behavior patterns, but you actually just re-discovered one column.

Treating clusters as permanent truth. Clusters are a lens, not a fact. If you re-run K-Means with a different random start and get totally different groupings, that instability means your "archetypes" aren't real, stable patterns — that's why the validation step for this lane is specifically a stability check: rerun it, see if the same groups survive.

Calling it "semantic" clustering. Your data has no article text — only numbers. If you claim your clusters understand what the content is about, that's a claim you can't back up, and it's explicitly flagged as a common mistake in your repo's own guide.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Section 2 is about decision → action → cost. The most honest thing you can back with code isn't a fancy calculation — it's just grounding the scale of the decision in a real number. Right now your cost argument is probably something like "if we mislabel a cluster, someone wastes time on the wrong pages" — that's a claim. A quick number turns it into something backed by the actual data: how many pages are we even talking about? """
print(np.shape(reading)) # Tells the number of web pages that we are talking about and the number of columns, 30K pages and 44 columns

(30000, 44)


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

engagement_rate — the median is literally 0. That means more than half of all 30,000 pages have zero engagement recorded. But the mean is 2.53, not 0 — meaning a smaller number of pages have very high engagement (up to 100), pulling the average up. Picture it like a company where most employees earn nothing extra, but 2 executives get huge bonuses — the "average bonus" looks decent, but it doesn't describe a single typical employee. That's your engagement_rate column.

ctr — same shape, slightly less extreme. Median is 0.07, but the 75th percentile is only 0.29 while the max is 100. So again: most pages sit near the bottom, with a small number of extreme outliers stretching the range enormously.

content_age_days — this one behaves normally. Mean (256) and median (236) are close, min is 90, max is 564 — a smooth, believable spread with no extreme skew. Content ages roughly evenly from ~3 months to ~1.5 years old.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
reading[["engagement_rate","ctr","content_age_days"]].describe()

,engagement_rate,ctr,content_age_days
count,30000.000000,30000.000000,30000.00000
mean,2.534520,0.510733,256.16780
std,8.310096,3.279162,132.70793
min,0.000000,0.000000,90.00000
25%,0.000000,0.000000,132.00000
50%,0.000000,0.070000,236.00000
75%,1.350000,0.290000,333.00000
max,100.000000,100.000000,564.00000


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

#This clustering will show observed behavioral groupings among pages based on current numeric signals — useful for prioritizing which pages a human reviews first. It will not explain why a page behaves the way it does, predict future performance, or describe what a page's content is actually about.

what clustering will prove 

That distinct behavioral groups exist in the data — e.g., "there's a large cluster of pages with near-zero engagement and stable rank, and a smaller cluster with high engagement and improving rank."

A prioritization tool — "here are 6 pages that look like Cluster 3 (declining, once-popular) — a human should look at these first."

A description of the present, based on the numbers you fed it (engagement, CTR, age, position, etc.)

A repeatable pattern, if it survives a stability check — meaning re-running it gives roughly the same groups.


What clustering will NOT prove (the traps to name explicitly)
It will NOT tell you why a page is in a certain state. Clustering never explains cause — it only shows "these pages behave alike." Whether that's because of bad content, a Google algorithm shift, or seasonal demand is completely outside what clustering can tell you.

It will NOT predict the future. A page being in the "declining" cluster today says nothing certain about what happens to it next month.

It will NOT understand what the page is about. You have no article text in this dataset — only numbers (engagement, CTR, age...). So you can never claim a cluster represents a topic or content type in a semantic sense — only a behavior pattern.

It will NOT hand you "correct" group names. K-Means doesn't know what "champions" or "stale pages" mean — you assign those labels after looking at what's actually inside each group, and even then, they're your interpretation, not ground truth.

It will NOT guarantee the same answer every time, unless you specifically check for that (the stability check we discussed) — different runs can produce different groupings, especially near the boundaries between clusters.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(reading.shape[0])  # scale this claim applies to: ~30,000 pages

30000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.